# Sesión 16 — Mecanismos de Atención y Transformers
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo V · Arquitecturas Avanzadas y AI Generativa**

## Marco del módulo

El Módulo IV te dio los tres primitivos canónicos del deep learning: MLP, CNN, RNN.
El Módulo V muestra qué ocurre cuando los llevamos más lejos — y cuando abandonamos
completamente sus sesgos inductivos a favor de la atención aprendida.

El arco a lo largo de cinco sesiones:
- **Sesión 16** — La atención es todo lo que necesitas: construir un encoder Transformer desde cero
- **Sesión 17** — Transformers *en* biomedicina: NLP clínico, ViT, Transformers de EEG
- **Sesión 18** — VAE y GAN: los dos modelos generativos profundos clásicos
- **Sesión 19** — Modelos de difusión y LLM: qué impulsa la IA generativa moderna
- **Sesión 20** — Aprendizaje autosupervisado y contrastivo: aprender sin etiquetas

## Objetivos de aprendizaje

1. Derivar la **atención de producto punto escalada** desde el formalismo Query-Key-Value.
2. Implementar **atención multi-cabeza** y comprender por qué ayudan múltiples cabezas.
3. Construir un **bloque encoder Transformer** completo desde cero en PyTorch.
4. Comprender las codificaciones posicionales y por qué son necesarias.
5. Implementar un **patch embedding de ViT** y aplicarlo a espectrogramas de EEG.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Vaswani, A. et al. (2017). Attention is all you need. *NeurIPS*. — Leer junto con el código. |
| ★★★ | Dosovitskiy, A. et al. (2021). An image is worth 16×16 words: transformers for image recognition at scale. *ICLR*. (ViT) |
| ★★☆ | Bahdanau, D., Cho, K. & Bengio, Y. (2015). Neural machine translation by jointly learning to align and translate. *ICLR*. — Precursor de la atención. |
| ★★☆ | Kostas, D. et al. (2020). BENDR: using transformers and a contrastive self-supervised paradigm for EEG representation learning. *Front. Hum. Neurosci.* |
| ★☆☆ | Transformer ilustrado: https://jalammar.github.io/illustrated-transformer/ |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import warnings; warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng    = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams.update({
    'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11,
})
print(f'Dispositivo: {device}')

## Parte 1 — Atención de producto punto escalada desde cero

Dadas las consultas (queries) $\mathbf{Q}\in\mathbb{R}^{n\times d_k}$, las llaves (keys)
$\mathbf{K}\in\mathbb{R}^{m\times d_k}$, y los valores $\mathbf{V}\in\mathbb{R}^{m\times d_v}$:

$$\text{Attention}(\mathbf{Q},\mathbf{K},\mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right)\mathbf{V}$$

El escalado por $\sqrt{d_k}$ evita la saturación del softmax cuando $d_k$ es grande.

In [ ]:
def atencion_producto_punto_escalada(Q, K, V, mask=None):
    """
    Q : (batch, cabezas, seq_q, d_k)
    K : (batch, cabezas, seq_k, d_k)
    V : (batch, cabezas, seq_k, d_v)
    Retorna: salida (batch, cabezas, seq_q, d_v), pesos (batch, cabezas, seq_q, seq_k)
    """
    d_k    = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)   # (B,H,q,k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    weights = F.softmax(scores, dim=-1)   # (B,H,q,k)  — las filas suman 1
    output  = torch.matmul(weights, V)    # (B,H,q,d_v)
    return output, weights


# ── Visualizar atención en una secuencia EEG de juguete ───────────────────────
# Secuencia: 10 épocas de EEG, vector de características de 8 dimensiones cada una
T_eeg, d_model = 10, 8
d_k = d_model

torch.manual_seed(0)
# Q = K = V (auto-atención)
X_demo = torch.randn(1, 1, T_eeg, d_k)   # una sola cabeza, un solo batch

_, attn_weights = atencion_producto_punto_escalada(X_demo, X_demo, X_demo)
attn_matrix = attn_weights[0, 0].detach().numpy()   # (T, T)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

im0 = axes[0].imshow(attn_matrix, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im0, ax=axes[0])
axes[0].set(xlabel='Posición llave (es atendida)',
            ylabel='Posición consulta (atiende desde)',
            title='Pesos de auto-atención\n(inicialización aleatoria, sin escalar)')

# Efecto del escalado
d_k_vals = [1, 4, 16, 64, 256]
entropia_escalada    = []
entropia_sin_escalar = []
for dk in d_k_vals:
    Q_test = torch.randn(1, 1, 20, dk)
    scores_u = torch.matmul(Q_test, Q_test.transpose(-2,-1))
    scores_s = scores_u / math.sqrt(dk)
    w_u = F.softmax(scores_u, dim=-1)
    w_s = F.softmax(scores_s, dim=-1)
    # Entropía de la distribución de atención (alta = difusa, baja = concentrada)
    ent_u = -(w_u * (w_u + 1e-9).log()).sum(-1).mean().item()
    ent_s = -(w_s * (w_s + 1e-9).log()).sum(-1).mean().item()
    entropia_sin_escalar.append(ent_u)
    entropia_escalada.append(ent_s)

axes[1].semilogx(d_k_vals, entropia_sin_escalar, 'r-o', lw=2, ms=8, label='Sin escalar')
axes[1].semilogx(d_k_vals, entropia_escalada,     'b-s', lw=2, ms=8, label='Escalada  ÷√d_k')
axes[1].set(xlabel='d_k', ylabel='Entropía de atención',
            title='Efecto del escalado √d_k\nSin escalar → concentrada (baja entropía) al crecer d_k')
axes[1].legend()

# Visualizar qué le hace la atención a una secuencia
x_seq = np.zeros((T_eeg, 2))
x_seq[:5,  0] = np.linspace(0, 1, 5)  # primera mitad
x_seq[5:,  1] = np.linspace(0, 1, 5)  # segunda mitad
X_seq_t = torch.tensor(x_seq, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
Q2 = K2 = V2 = X_seq_t.expand(-1, -1, -1, -1)
attn_out, _ = atencion_producto_punto_escalada(
    X_seq_t, X_seq_t, X_seq_t)
axes[2].imshow(attn_matrix, cmap='YlOrRd', aspect='auto')
axes[2].set(xlabel='Posición', ylabel='Posición',
            title='Patrón de la matriz de atención\n(cada fila = a dónde mira esta posición)')

plt.tight_layout()
plt.show()

## Parte 2 — Atención multi-cabeza

En lugar de una sola función de atención, se proyecta en $h$ subespacios y se atiende
en paralelo:

$$\text{MultiHead}(\mathbf{Q},\mathbf{K},\mathbf{V}) = \text{Concat}(\text{head}_1,\ldots,\text{head}_h)\mathbf{W}^O$$
$$\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}_i^Q,\, \mathbf{K}\mathbf{W}_i^K,\, \mathbf{V}\mathbf{W}_i^V)$$

Cada cabeza puede aprender a atender a patrones distintos de la secuencia.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model debe ser divisible por n_heads'
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        """(B, T, d_model) → (B, h, T, d_k)"""
        B, T, _ = x.shape
        return x.reshape(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, Q, K, V, mask=None):
        B = Q.size(0)
        Q = self.split_heads(self.W_Q(Q))  # (B, h, T, d_k)
        K = self.split_heads(self.W_K(K))
        V = self.split_heads(self.W_V(V))

        attn_out, attn_w = atencion_producto_punto_escalada(Q, K, V, mask)
        attn_out = self.dropout(attn_out)

        # Fusionar cabezas: (B, h, T, d_k) → (B, T, d_model)
        attn_out = attn_out.transpose(1, 2).contiguous().reshape(B, -1, self.d_model)
        return self.W_O(attn_out), attn_w


# Probar la atención multi-cabeza
mha = MultiHeadAttention(d_model=64, n_heads=8)
x_test_mha = torch.randn(4, 20, 64)   # batch=4, seq=20, d_model=64
out_mha, w_mha = mha(x_test_mha, x_test_mha, x_test_mha)
print(f'Entrada MHA:  {tuple(x_test_mha.shape)}')
print(f'Salida MHA:   {tuple(out_mha.shape)}  (misma forma — la atención preserva el tamaño)')
print(f'Forma pesos de atención: {tuple(w_mha.shape)}  (batch, cabezas, seq_q, seq_k)')

# Visualizar las 8 cabezas en una secuencia de 20 tokens
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for head_idx, ax in enumerate(axes.flat):
    heatmap = w_mha[0, head_idx].detach().numpy()   # (20, 20)
    im = ax.imshow(heatmap, cmap='Blues', vmin=0, vmax=heatmap.max())
    ax.set(title=f'Cabeza {head_idx+1}', xticks=[], yticks=[])

plt.suptitle('Auto-atención multi-cabeza — 8 cabezas aprenden patrones distintos\n'
             'Cada cabeza atiende a aspectos diferentes de la secuencia', y=1.01)
plt.tight_layout()
plt.show()

## Parte 3 — Bloque encoder Transformer completo

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Codificación posicional sinusoidal de Vaswani et al. (2017).
    PE(pos, 2i)   = sin(pos / 10000^{2i/d_model})
    PE(pos, 2i+1) = cos(pos / 10000^{2i/d_model})
    """
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() *
                         (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerEncoderBlock(nn.Module):
    """
    Bloque encoder Transformer con Pre-LayerNorm (variante moderna, más estable que la original).
    x → LayerNorm → MHA → residual → LayerNorm → FFN → residual
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask=None):
        # Subcapa de auto-atención (pre-LN)
        attn_out, attn_w = self.attn(self.norm1(x), self.norm1(x), self.norm1(x), mask)
        x = x + attn_out
        # Subcapa feed-forward (pre-LN)
        x = x + self.ffn(self.norm2(x))
        return x, attn_w


class TransformerEncoder(nn.Module):
    """Pila de N bloques encoder Transformer con codificación posicional."""
    def __init__(self, d_model, n_heads, d_ff, n_layers, dropout=0.1, max_len=512):
        super().__init__()
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.layers  = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.pos_enc(x)
        attn_weights = []
        for layer in self.layers:
            x, attn_w = layer(x, mask)
            attn_weights.append(attn_w)
        return self.norm(x), attn_weights


# Verificación rápida
enc = TransformerEncoder(d_model=64, n_heads=8, d_ff=256, n_layers=4)
x_in = torch.randn(4, 20, 64)
x_out, ws = enc(x_in)
print(f'Entrada del encoder:  {tuple(x_in.shape)}')
print(f'Salida del encoder:   {tuple(x_out.shape)}  (T × d_model preservado)')
print(f'Pesos de atención por capa: {[tuple(w.shape) for w in ws]}')

# Visualizar la codificación posicional
pe_module = PositionalEncoding(d_model=128, max_len=100)
pe_matrix = pe_module.pe[0].numpy()   # (100, 128)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
im = axes[0].imshow(pe_matrix.T, aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[0])
axes[0].set(xlabel='Posición', ylabel='Dimensión del embedding',
            title='Codificación posicional sinusoidal (128-dim, 100 posiciones)\n'
                  'Cada fila es una "huella digital" posicional única')

# Similitud por producto punto entre posiciones
pe_norm = pe_matrix / (np.linalg.norm(pe_matrix, axis=1, keepdims=True) + 1e-8)
sim_mat  = pe_norm @ pe_norm.T
im2 = axes[1].imshow(sim_mat, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im2, ax=axes[1])
axes[1].set(xlabel='Posición j', ylabel='Posición i',
            title='Similitud posicional (producto punto)\n'
                  'Posiciones cercanas son más similares → contexto local')

plt.tight_layout()
plt.show()

## Parte 4 — Transformer de EEG para detección de crisis

Aplicamos el encoder como un clasificador de secuencias con un **token [CLS]**
(estilo BERT).

In [ ]:
class EEGTransformer(nn.Module):
    """
    Encoder Transformer para clasificación de secuencias de EEG.
    Entrada: (batch, T, n_features) — secuencia de ventanas de características EEG
    Usa un token [CLS] aprendible anteponiéndolo a la secuencia;
    la salida del CLS se usa para la clasificación (estilo BERT).
    """
    def __init__(self, n_features, d_model=64, n_heads=8, d_ff=256,
                 n_layers=3, dropout=0.1, n_classes=2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.cls_token  = nn.Parameter(torch.randn(1, 1, d_model))
        self.encoder    = TransformerEncoder(d_model, n_heads, d_ff, n_layers, dropout)
        self.head       = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, x):
        B, T, _ = x.shape
        x = self.input_proj(x)                          # (B, T, d_model)
        cls = self.cls_token.expand(B, -1, -1)          # (B, 1, d_model)
        x   = torch.cat([cls, x], dim=1)                # (B, T+1, d_model)
        x, attn_ws = self.encoder(x)                    # (B, T+1, d_model)
        cls_out = x[:, 0]                               # (B, d_model) — token CLS
        return self.head(cls_out), attn_ws              # (B, n_classes)


# ── Simular dataset de crisis EEG (igual que en la Sesión 14) ────────────────
def simular_ventanas_eeg(n_subj=20, T=60, n_feat=8, seed=0):
    rng_l = np.random.default_rng(seed)
    X_all, y_all = [], []
    for s in range(n_subj):
        X  = rng_l.normal(0, 1, (T, n_feat)).astype(np.float32)
        y  = np.zeros(T, dtype=np.int64)
        st = rng_l.integers(20, 40)
        dur = rng_l.integers(10, 20)
        for w in range(st, min(st+dur, T)):
            X[w, :3] += rng_l.normal(2.0, 0.5, 3)
            y[w] = 1
        X_all.append(X); y_all.append(y)
    return np.stack(X_all), np.stack(y_all)

X_eeg_s, y_eeg_s = simular_ventanas_eeg()
# Normalizar
mu = X_eeg_s.mean((0,1), keepdims=True)
sd = X_eeg_s.std((0,1),  keepdims=True) + 1e-8
X_eeg_s = (X_eeg_s - mu) / sd

# División LOSO
Xtr_e = torch.tensor(X_eeg_s[:16], dtype=torch.float32)
ytr_e = torch.tensor(y_eeg_s[:16], dtype=torch.long)
Xte_e = torch.tensor(X_eeg_s[16:], dtype=torch.float32)
yte_e = torch.tensor(y_eeg_s[16:], dtype=torch.long)

# NOTA: para una tarea a nivel de secuencia, usamos las etiquetas por ventana directamente
# Reorganizamos a (N_ventanas, T_contexto, características) con ventana deslizante de contexto
ctx = 10   # ventana de contexto — el Transformer ve 10 épocas a la vez

def crear_ventanas(X, y, ctx=10):
    Xw, yw = [], []
    B, T, F = X.shape
    for b in range(B):
        for t in range(ctx, T):
            Xw.append(X[b, t-ctx:t])   # ventana de contexto
            yw.append(y[b, t-1].item())
    return torch.stack(Xw), torch.tensor(yw, dtype=torch.long)

Xw_tr, yw_tr = crear_ventanas(Xtr_e, ytr_e, ctx)
Xw_te, yw_te = crear_ventanas(Xte_e, yte_e, ctx)

print(f'Train con ventanas: {Xw_tr.shape}  etiquetas: {yw_tr.shape}')
print(f'Test con ventanas:  {Xw_te.shape}')

tr_loader = DataLoader(TensorDataset(Xw_tr, yw_tr), batch_size=128, shuffle=True)

# Entrenar
eeg_transformer = EEGTransformer(n_features=8, d_model=64, n_heads=8,
                                   d_ff=256, n_layers=3, n_classes=2).to(device)
n_params = sum(p.numel() for p in eeg_transformer.parameters())
print(f'EEGTransformer: {n_params:,} parámetros')

opt_tr = optim.AdamW(eeg_transformer.parameters(), lr=3e-4, weight_decay=1e-2)
sched  = optim.lr_scheduler.CosineAnnealingLR(opt_tr, T_max=40)
pos_w  = (yw_tr == 0).sum().float() / (yw_tr == 1).sum().float()
crit_t = nn.CrossEntropyLoss(weight=torch.tensor([1.0, pos_w.item()]).to(device))

hist_tr = []
for ep in range(40):
    eeg_transformer.train()
    ep_loss = 0
    for Xb, yb in tr_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        opt_tr.zero_grad()
        logits, _ = eeg_transformer(Xb)
        loss = crit_t(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(eeg_transformer.parameters(), 1.0)
        opt_tr.step()
        ep_loss += loss.item()
    sched.step()
    hist_tr.append(ep_loss / len(tr_loader))
    if (ep+1) % 10 == 0:
        eeg_transformer.eval()
        with torch.no_grad():
            p = F.softmax(eeg_transformer(Xw_te.to(device))[0], dim=1)[:,1].cpu().numpy()
        auroc = roc_auc_score(yw_te.numpy(), p)
        print(f'Época {ep+1:3d}  pérdida={hist_tr[-1]:.4f}  AUROC={auroc:.4f}')

## Parte 5 — Vision Transformer (ViT): patch embedding para espectrogramas de EEG

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Patch embedding estilo ViT para espectrogramas 2-D.
    Divide la imagen (H, W) en parches (patch_h × patch_w) y proyecta cada uno a d_model.
    """
    def __init__(self, img_h, img_w, patch_h, patch_w, in_channels, d_model):
        super().__init__()
        assert img_h % patch_h == 0 and img_w % patch_w == 0
        self.n_patches = (img_h // patch_h) * (img_w // patch_w)
        self.patch_h   = patch_h
        self.patch_w   = patch_w
        # Una sola convolución con stride = tamaño de parche implementa extracción + proyección lineal
        self.proj = nn.Conv2d(in_channels, d_model,
                               kernel_size=(patch_h, patch_w),
                               stride=(patch_h, patch_w))

    def forward(self, x):
        # x: (B, C, H, W)
        x = self.proj(x)             # (B, d_model, H/ph, W/pw)
        x = x.flatten(2)             # (B, d_model, n_parches)
        x = x.transpose(1, 2)        # (B, n_parches, d_model)
        return x


class ViT_EEG(nn.Module):
    """
    Vision Transformer para clasificación de espectrogramas de EEG.
    Entrada: (B, 1, bins_frecuencia, marcos_tiempo) — espectrograma de un solo canal
    """
    def __init__(self, img_h=64, img_w=64, patch_h=8, patch_w=8,
                 d_model=128, n_heads=8, d_ff=256, n_layers=4,
                 n_classes=2, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_h, img_w, patch_h, patch_w, 1, d_model)
        n_patches = (img_h // patch_h) * (img_w // patch_w)

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches+1, d_model) * 0.02)
        self.encoder   = TransformerEncoder(d_model, n_heads, d_ff, n_layers, dropout)
        self.head      = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, n_classes)
        )

    def forward(self, x):
        B = x.size(0)
        tokens = self.patch_embed(x)                    # (B, n_p, d_model)
        cls    = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)        # (B, n_p+1, d_model)
        tokens = tokens + self.pos_embed                # embedding posicional aprendible
        tokens, _ = self.encoder(tokens)
        return self.head(tokens[:, 0])                  # token CLS


# Prueba
vit = ViT_EEG(img_h=64, img_w=64, patch_h=8, patch_w=8, d_model=128, n_layers=4)
spec_batch = torch.randn(4, 1, 64, 64)   # 4 espectrogramas
out_vit    = vit(spec_batch)
n_p = (64//8)**2
print(f'ViT-EEG:')
print(f'  Espectrograma de entrada: {tuple(spec_batch.shape)}')
print(f'  Cuadrícula de parches:    {64//8} × {64//8} = {n_p} parches de 8×8 píxeles')
print(f'  Secuencia al Transformer: {n_p}+1 (CLS) = {n_p+1} tokens de dim 128')
print(f'  Salida:                   {tuple(out_vit.shape)}')
print(f'  Parámetros:               {sum(p.numel() for p in vit.parameters()):,}')

# Visualizar la división en parches sobre un espectrograma EEG sintético
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

spec_demo = rng.random((64, 64))
# Añadir algo de estructura
spec_demo[20:35, 15:50] += 1.5  # ráfaga ictal simulada
spec_demo = (spec_demo - spec_demo.min()) / (spec_demo.max() - spec_demo.min())

axes[0].imshow(spec_demo, aspect='auto', cmap='inferno', origin='lower')
# Dibujar la cuadrícula de parches
for i in range(0, 64, 8):
    axes[0].axhline(i, color='white', lw=0.8, alpha=0.7)
    axes[0].axvline(i, color='white', lw=0.8, alpha=0.7)
axes[0].set(xlabel='Marcos de tiempo', ylabel='Bins de frecuencia',
            title=f'Espectrograma EEG con cuadrícula de parches 8×8\n({n_p} parches → {n_p} tokens)')

# Mostrar algunos parches como tokens
patch_tokens = []
for pi in range(0, 64, 8):
    for pj in range(0, 64, 8):
        patch_tokens.append(spec_demo[pi:pi+8, pj:pj+8].ravel())
patch_tokens = np.array(patch_tokens)   # (64, 64)
axes[1].imshow(patch_tokens, aspect='auto', cmap='viridis')
axes[1].set(xlabel='Píxel dentro del parche (aplanado, 64-dim)',
            ylabel='Índice de parche (64 parches)',
            title='Tokens de parche antes de la proyección lineal\n'
                  'Cada fila = un parche 8×8 aplanado a 64 dims')

plt.tight_layout()
plt.show()

## ✏️ Ejercicios

1. **Máscara causal.** Añade una máscara causal (triangular superior) a la atención para
   que la posición $t$ solo pueda atender a posiciones $\le t$. Verifica que el modelo
   causal coincide con el patrón de generación autoregresiva. ¿Cuándo se prefiere la
   atención causal sobre la bidireccional en contextos biomédicos?

2. **Codificación posicional relativa.** Implementa RoPE (Rotary Position Embedding)
   como alternativa a la codificación posicional sinusoidal. Compara las curvas de
   entrenamiento y el AUROC de clasificación en la tarea de crisis EEG. ¿Por qué las
   codificaciones relativas podrían superar a las absolutas en grabaciones de EEG de
   longitud variable?

3. **Visualización de la atención.** Después de entrenar el EEGTransformer, extrae los
   pesos de atención de cada capa y cabeza. Grafica el peso medio de atención desde el
   token CLS hacia cada posición de entrada, para ventanas de crisis correctamente e
   incorrectamente clasificadas. ¿Los patrones de atención aprendidos se alinean con
   características conocidas del EEG ictal?

4. **ViT en espectrogramas de EEG.** Genera imágenes de espectrograma de 64×64
   (frecuencia × tiempo) a partir de la base de datos CHB-MIT EEG usando
   `mne.time_frequency.psd_array_multitaper` de MNE-Python. Entrena el ViT-EEG y compara
   el AUROC en test con el BiLSTM de la Sesión 14. ¿Qué arquitectura es más eficiente
   en datos?

5. *(Desafío)* **Intuición de Flash Attention.** La atención estándar tiene complejidad
   de memoria $O(T^2)$. Implementa una aproximación eficiente en memoria usando atención
   de características aleatorias ($\text{softmax}(\mathbf{QK}^\top) \approx
   \phi(\mathbf{Q})\phi(\mathbf{K})^\top$ con características aleatorias de Fourier) y
   compara velocidad y exactitud en secuencias de longitud 512, 1024, 2048.

## 📚 Conjuntos de datos

| Conjunto de datos | Fuente | Notas |
|---|---|
| CHB-MIT EEG | https://physionet.org/content/chbmit/ | Referencia de detección de crisis |
| Temple Univ EEG Corpus | https://isip.piconepress.com/projects/tuh_eeg/ | El corpus público de EEG más grande |
| PhysioNet PTB-XL | https://physionet.org/content/ptb-xl/ | ECG de 12 derivaciones, 21k pacientes |